In [ ]:
import sys
from pathlib import Path

racine_projet = Path.cwd()
if not (racine_projet / "commun").is_dir():
    racine_projet = racine_projet.parent
if not (racine_projet / "commun").is_dir():
    raise FileNotFoundError("Exécutez ce notebook depuis le dossier work du projet.")
sys.path.insert(0, str(racine_projet))
from commun.spark_utils import BUCKET, LAKE, creer_bucket, creer_session, format_tables

creer_bucket(BUCKET)

In [ ]:
spark = creer_session("S01b")
FMT = format_tables(spark)
if FMT != "delta":
    raise RuntimeError("Delta Lake indisponible. Utilisez l'environnement Delta isolé décrit dans le README.")
print("Format :", FMT)

In [ ]:
fichier = racine_projet / "data/raw/comptage_multimodal_comptages.csv"
if not fichier.is_file():
    raise FileNotFoundError(f"CSV introuvable : {fichier}. Lancez le script de téléchargement.")
bronze_path = f"{LAKE}/bronze/comptage_multimodal_paris"
path = spark._jvm.org.apache.hadoop.fs.Path(bronze_path)
if path.getFileSystem(spark._jsc.hadoopConfiguration()).exists(path):
    raise FileExistsError(f"Bronze existe déjà : {bronze_path}. Utilisez un environnement isolé.")

comptages_raw = (
    spark.read
    .option("header", "true")
    .option("sep", ";")
    .option("mode", "FAILFAST")
    .csv(str(fichier))
)

nombre_source = comptages_raw.count()
(comptages_raw.write.format("delta").mode("errorifexists")
    .option("delta.columnMapping.mode", "name")
    .option("delta.minReaderVersion", "2")
    .option("delta.minWriterVersion", "5")
    .save(bronze_path))
bronze = spark.read.format("delta").load(bronze_path)
nombre_bronze = bronze.count()
if bronze.columns != comptages_raw.columns or nombre_bronze != nombre_source:
    raise RuntimeError("La vérification du nombre de lignes ou des colonnes Bronze a échoué.")
print(f"Bronze vérifié : {nombre_bronze} lignes, colonnes conservées : {bronze.columns}")